# Delivery Route Optimisation Using ILP and SA

IT5082 Optimization Methods

**Problem:**  Capacitated Vehicle Routing Problem (CVRP) <br>
**Exact method:** ILP <br>
**Heuristic:**  Simulated Annealing (SA) <br>
**Data:**  CVRPLIB benchmarks and the Amazon Last Mile 2021 routes<br>
**M L I U Meegahawala (MS26903538)  / M T M Ibjas (MS25942248)**

## Setup
Install the packages from requirements.txt

In [5]:
import itertools, json, math, os, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import milp, linprog, LinearConstraint, Bounds
from scipy.sparse import coo_matrix
import vrplib

# numba speedup the  SA decoder
try:
    from numba import njit
except ImportError:
    def njit(*args, **kwargs):
        return args[0] if args and callable(args[0]) else (lambda f: f)


SEED = 42
CONFIG = dict(
    exact_time_limit = 120,    # seconds per ILP solve
    exact_max_n      = 45,     # largest instance we try with the ILP
    n_seeds          = 10,     # SA runs per instance
    sa_iters_per_n   = 1500,   # SA iterations = this x number of customers
)

def sa_budget(inst):
    """SA iterations for an instance. The same for every seed, so the comparison is fair."""
    return CONFIG["sa_iters_per_n"] * inst["n"]


# Folders: raw input files, saved instances, figures. Created if they do not exist yet.
RAW, INST, FIG = Path("data/raw"), Path("data/instances"), Path("results/figures")
for p in (RAW, INST, FIG, Path("results")):
    p.mkdir(parents=True, exist_ok=True)

# Plot style for every figure: default size, light grid, no top and right border lines
plt.rcParams.update({"figure.figsize": (7, 4.5), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

# Fixed random seeds, so a re-run gives the same numbers
random.seed(SEED)
np.random.seed(SEED)

print("Config:", CONFIG)

Config: {'exact_time_limit': 120, 'exact_max_n': 45, 'n_seeds': 10, 'sa_iters_per_n': 1500}
